# 05 - Longitudinal impedance and wake

Wake elements are collective: they act on an `N x 6` bunch through `linepass!`, not on a single `SVector`. Positive `z` means an early particle.

In [ ]:
import Pkg
EXAMPLES_DIR = isfile(joinpath(pwd(), "common.jl")) ? pwd() : joinpath(pwd(), "examples")
Pkg.activate(EXAMPLES_DIR)
using TrackPad, StaticArrays
include(joinpath(EXAMPLES_DIR, "common.jl"))
using .TrackPadExamples

using CairoMakie, Random

In [ ]:
ring, beam = madx_fodo()
N = 20_000
coordinates = zeros(N, 6)
coordinates[:, 5] .= 3e-3 .* randn(MersenneTwister(42), N)
initial_delta = copy(coordinates[:, 6])

scale = physical_wake_scale(beam, -1e-9, N)
wake = LongitudinalRLCWake(
    freq=1.0e9, Rshunt=1.0e6, Q0=2.0, scale=scale, nbins=128,
)
lost = zeros(Int, N)
wake_ring = Lattice(vcat(ring.elements, AbstractElement[wake]); periodic=true)
linepass!(coordinates, wake_ring, beam, lost)


In [ ]:
fig = Figure(size=(700, 330))
ax = Axis(fig[1, 1], xlabel="z [mm]", ylabel="Delta delta_E")
scatter!(ax, 1e3 .* coordinates[1:20:end, 5],
         coordinates[1:20:end, 6] .- initial_delta[1:20:end]; markersize=3)
fig

`LongitudinalRLCWake` and `LongitudinalWake` store causal Green functions. TrackPad deposits the bunch with cloud-in-cell weights, convolves the histogram, and interpolates the potential back to each particle. `physical_wake_scale` supplies charge and `P0*c` normalization for a Green function in V/C.